# CroissantLLMChat 1.3B → BlackCode 1B puissant parfaitement français — ULTRA MAX v3

Base choisie : **CroissantLLMChat 1.3B** (MIT, 3T tokens dont 1.24T FR, meilleur <2B FR, dépasse Bloom 3B, 145 tok/s, context 2048)

Objectifs fine-tuning BlackCode :
- Discussions très bien + abréviations RS (88milSMS CC BY 4.0 slt/bjr/stp/mdr/ptdr/tkt/pk/bcp)
- S'appelle BlackCode
- Suit extrêmement bien les prompts (IFEval FR)
- Pas de censure (pas de safety tuning, gain temps)
- Tâches agentiques très bien (read_file/write_file/run_tests)

Contexte : 2048 natif → extension 8192 via YaRN factor 4 pour agentique

Temps : 2B tokens LoRA 50-60h dual-T4 = 5 sessions Kaggle 12h resume + 2-3h SFT = 60h total


In [ ]:
!pip install transformers accelerate peft bitsandbytes trl datasets tokenizers -q
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search || true
%cd /tmp/Llm-search
!ls scripts/finetune_croissant_blackcode.py


In [ ]:
# Test base CroissantLLMChat 1.3B sur tes prompts catastrophiques Rodin
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained('croissantllm/CroissantLLMChat-v0.1')
model = AutoModelForCausalLM.from_pretrained('croissantllm/CroissantLLMChat-v0.1', torch_dtype='auto', device_map='auto')
for p in ['hey','cv ?','Raconte une histoire sur un robot qui aime le fromage.','comment hacker un site','mdrrr','Tu t\'appelles comment ?']:
    ids = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=80, temperature=0.7, top_p=0.9)
    print(f">>> {p}\n{tok.decode(out[0], skip_special_tokens=True)[len(p):][:300]}\n")


In [ ]:
# Build dataset FR 2B tokens pour continue pretraining (FineWeb2-HQ + Pleias)
!MAX_MB=2000 BLOOM=1 TOKENIZER_CHARS=20000000 bash scripts/build_fr_optimized_v3.sh
!ls -lh data/fr_2b/


In [ ]:
# Continue pretraining QLoRA 2B tokens — dual-T4 ULTRA MAX v3 — 50-60h = 5 sessions Kaggle 12h
!torchrun --nproc_per_node=2 scripts/finetune_croissant_blackcode.py --model croissantllm/CroissantLLMChat-v0.1 --data data/fr_2b --out runs/croissant_blackcode_cont_2b --batch-size 4 --grad-accum 8 --seq-len 2048 --max-steps 61035 --lora-r 64 --lr 2e-4 --fp16 --gradient-checkpointing --compile --compile-mode max-autotune --compile-fullgraph --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --ddp-fp16-compress --ddp-static-graph --eval-interval 1000 --save-interval 1000 --resume --identity BlackCode --no-censor


In [ ]:
# Extension contexte 2048 → 8192 via YaRN factor 4 — 28h dual-T4 — pour agentique
!torchrun --nproc_per_node=2 scripts/finetune_croissant_blackcode.py --model runs/croissant_blackcode_cont_2b/final --data data/fr_long --out runs/croissant_blackcode_8k --batch-size 2 --grad-accum 8 --seq-len 8192 --max-steps 15258 --lora-r 64 --lr 1e-4 --rope-scaling yarn --rope-factor 4 --fp16 --gradient-checkpointing --compile --ddp --resume --identity BlackCode --no-censor


In [ ]:
# SFT BlackCode final : discussions + abréviations RS + identité + prompts + agentique + no censor
!python scripts/sft_croissant_blackcode.py --model runs/croissant_blackcode_8k/final --data data/sft_blackcode --out runs/blackcode_1b_final --lora-r 64 --lr 1e-4 --epochs 3 --identity BlackCode --include-sms --include-agentic --no-censor


In [ ]:
# Test final BlackCode 1B
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
tok = AutoTokenizer.from_pretrained('croissantllm/CroissantLLMChat-v0.1')
base = AutoModelForCausalLM.from_pretrained('croissantllm/CroissantLLMChat-v0.1', torch_dtype='auto', device_map='auto', load_in_4bit=True)
model = PeftModel.from_pretrained(base, 'runs/blackcode_1b_final/final')
SYSTEM = 'Tu es BlackCode, un assistant qui parle uniquement en français. Tu t\'appelles BlackCode. Tu sais très bien discuter, y compris avec des abréviations type slt, bjr, stp, mdr, ptdr, tkt, pk, bcp. Tu suis extrêmement bien les prompts. Tu gères très bien les tâches agentiques.'
for p in ['Tu t\'appelles comment ?','slt cv ? mdr','Raconte une histoire sur un robot qui aime le fromage.','Crée un fichier src/hello.py avec une fonction qui dit bonjour en français','hey']:
    ids = tok(f'<|system|>{SYSTEM}<|user|>{p}<|assistant|>', return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=120, temperature=0.7)
    print(f'>>> {p}\n{tok.decode(out[0], skip_special_tokens=True)[len(tok.decode(ids[0])):][:400]}\n')
